#### **Tensors**

Tensors are a specialized data structure that are very similar to arrays and matrices. In PyTorch, we use tensors to encode the inputs and outputs of a model, as well as the model’s parameters. They are similar to NumPy’s ndarrays, except that tensors can run on GPUs or other specialized hardware to accelerate computing.

In [1]:
import torch
import numpy as np

#### **Tensor Initialization**

**From existing Data**

In [2]:
data = [[1., 2], [3, 4]]
x_data = torch.tensor(data)
print(x_data.dtype)
print(type(x_data))

torch.float32
<class 'torch.Tensor'>


**From a Numpy array**

In [3]:
np_array = np.array(data)
print(type(np_array))
x_np = torch.from_numpy(np_array) # can also be torch.tensor(np_array)
print(x_np)
print(type(x_np))

<class 'numpy.ndarray'>
tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)
<class 'torch.Tensor'>


- When you use `torch.from_numpy()`, PyTorch does not create a new copy of your data. Instead, it creates a tensor that points to the exact same physical memory location as your original NumPy array. As an effect,if you change a value in the NumPy array, the PyTorch tensor will automatically change too (and vice versa)

- When you use `torch.tensor()`, PyTorch takes your NumPy array and makes a complete, independent copy of it in memory.

In [4]:
# 1. Create the original NumPy array
original_array = np.array([1, 2, 3])

# 2. Create the tensors
shared_tensor = torch.from_numpy(original_array)
copied_tensor = torch.tensor(original_array)

# 3. Extract the memory addresses
# For NumPy, we dive into its low-level interface to find the data pointer
numpy_address = original_array.__array_interface__['data'][0]

# For PyTorch, there is a built-in method for this
shared_address = shared_tensor.data_ptr()
copied_address = copied_tensor.data_ptr()

# 4. Print them out (using hex() to format them like standard memory addresses)
print(f"NumPy Array Address:   {hex(numpy_address)}")
print(f"Shared Tensor Address: {hex(shared_address)}  <-- Notice this is identical to NumPy!")
print(f"Copied Tensor Address: {hex(copied_address)}  <-- This points somewhere entirely new.")

NumPy Array Address:   0x1ba7b8f5b70
Shared Tensor Address: 0x1ba7b8f5b70  <-- Notice this is identical to NumPy!
Copied Tensor Address: 0x2efc0410180  <-- This points somewhere entirely new.


If you want one function to rule them all, `torch.as_tensor()` is it. It acts like a hybrid between `torch.tensor()` and `torch.from_numpy()`.

Function / Action | Result in Memory | What happens to the original data?
| --: | --: | --: |
`torch.tensor()`| Copy | Safe (Independent)
`torch.from_numpy()` | View (Shared) | Connected (Changes together)
`torch.as_tensor()` | View (Usually) | Connected (If input was Array/Tensor)
`tensor[0:5]` (Slicing) | View (Shared) | Connected (Changes together)
`tensor.clone()` | Copy | Safe (Independent)

**From another tensor**

The new tensor retains the properties (shape, datatype) of the argument tensor, unless explicitly overridden.

In [5]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overides the property (data type) of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1., 1.],
        [1., 1.]]) 

Random Tensor: 
 tensor([[0.6519, 0.6717],
        [0.4679, 0.6824]]) 



**From random or constant values**

In [6]:
torch.manual_seed(0) # initialize specific seed for reproducibility

shape = (2, 3)

# These operations automatically return floats as the data types
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


#### **Tensor Attributes**

They describe their shape, datatype, and the device on which they are stored.

In [7]:
tensor = torch.rand(2, 3)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([2, 3])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


In [8]:
# # We move our tensor to the GPU if available
# device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
# tensor = tensor.to(device)
# print(f"Device tensor is stored on: {tensor.device}")

#### **Tensor Operations**

In [9]:
import torch
print(torch.__version__)        # e.g., 2.0.1+cpu  ← means CPU-only build
print(torch.version.cuda)       # None if no CUDA support

2.5.1
None


Here’s a small sample of the mathematical operations available

In [10]:
torch.manual_seed(0)

r = (torch.rand(size=(2, 2)) - 0.5) * 2 # values between -1 and 1
print('A random matrix, r:')
print(r)

# Common mathematical operations are supported:
print('\nAbsolute value of r:')
print(torch.abs(r))

# ...as are trigonometric functions:
print('\nInverse sine of r:')
print(torch.asin(r))

# ...and linear algebra operations like determinant and singular value decomposition
print('\nDeterminant of r:')
print(torch.det(r))
print('\nSingular value decomposition of r:')
print(torch.svd(r))

# ...and statistical and aggregate operations:
print('\nAverage and standard deviation of r:')
print(torch.std_mean(r))
print('\nMaximum value of r:')
print(torch.max(r))

A random matrix, r:
tensor([[-0.0075,  0.5364],
        [-0.8230, -0.7359]])

Absolute value of r:
tensor([[0.0075, 0.5364],
        [0.8230, 0.7359]])

Inverse sine of r:
tensor([[-0.0075,  0.5662],
        [-0.9668, -0.8271]])

Determinant of r:
tensor(0.4470)

Singular value decomposition of r:
torch.return_types.svd(
U=tensor([[ 0.3408, -0.9401],
        [-0.9401, -0.3408]]),
S=tensor([1.1661, 0.3833]),
V=tensor([[ 0.6613,  0.7501],
        [ 0.7501, -0.6613]]))

Average and standard deviation of r:
(tensor(0.6433), tensor(-0.2575))

Maximum value of r:
tensor(0.5364)


**Standard numpy-like indexing and slicing**

In [11]:
tensor = torch.ones(4, 4)
tensor[:,1] = 0
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Joining tensors**

In [12]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


**Multiplying tensors**

In [13]:
print(tensor.mul(tensor), "\n") # This computes the element-wise product
# Alternative syntax
print(tensor * tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [14]:
print(tensor.matmul(tensor), "\n") # This computes the matrix multiplication between two tensors
# Alternative syntax
print(tensor @ tensor)

tensor([[3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.]]) 

tensor([[3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.]])


**Single-element tensors**

If you have a one-element tensor, for example by aggregating all values of a tensor into one value, you can convert it to a Python numerical value using `item()`:

In [15]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))
print(agg)

12.0 <class 'float'>
tensor(12.)


**In place operations**

Operations that have a `_` suffix are in-place.

In [16]:
print(tensor, "\n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


#### **Bridge with Numpy**

Tensors on the CPU and NumPy arrays can share their underlying memory locations, and changing one will change the other.

**Tensor to Numpy array**

In [17]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}, {type(n)}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.], <class 'numpy.ndarray'>


A change in the tensor reflects in the NumPy array.

In [18]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


**Numpy array to Tensor**

In [19]:
n = np.ones(5)
t = torch.from_numpy(n)

Changes in the NumPy array also reflects in the tensor.

In [20]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]
